In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import os
os.chdir('/content/gdrive/My Drive/MARS')
print("✅ Working directory:", os.getcwd())

Mounted at /content/gdrive
✅ Working directory: /content/gdrive/My Drive/MARS


In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [ ]:
import pandas as pd

df = pd.read_csv(
    "feature_engineered.csv"
)

print(df.shape)

(20000, 14)


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def build_prompt(
    subject,
    description,
    category,
    channel
):

    return f"""
You are a support ticket severity auditor.

Assign a severity score.

Severity scale:

0 = Low
1 = Medium
2 = High
3 = Critical

Guidelines:

3:
Security incidents,
fraud,
service outages,
major operational disruption.

2:
Major functionality failures,
API failures,
application crashes,
login failures,
payment issues.

1:
User-impacting issues with workarounds.

0:
Questions,
clarifications,
information requests,
minor requests.

Ticket Category:
{category}

Ticket Channel:
{channel}

Subject:
{subject}

Description:
{description}

Return ONLY ONE NUMBER.

Severity Score:
"""

In [ ]:
import re

def predict_severity(
    subject,
    description,
    category,
    channel
):

    prompt = build_prompt(
        subject,
        description,
        category,
        channel
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    numbers = re.findall(
        r"\b([0-3])\b",
        response
    )

    if len(numbers) > 0:
        return int(numbers[0])

    return None

In [ ]:
sample_df = df.sample(
    500,
    random_state=42
)

In [ ]:
scores = []

for _, row in sample_df.iterrows():

    score = predict_severity(
        row["Ticket_Subject"],
        row["Ticket_Description"],
        row["Issue_Category"],
        row["Ticket_Channel"]
    )

    scores.append(score)

sample_df["llm_score"] = scores

In [ ]:
print(
    sample_df["llm_score"]
    .value_counts(dropna=False)
)

llm_score
0    297
2    140
1     33
3     30
Name: count, dtype: int64


In [ ]:
sample_df[
    [
        "Ticket_Subject",
        "Issue_Category",
        "Priority_Level",
        "llm_score"
    ]
].head(20)

,Ticket_Subject,Issue_Category,Priority_Level,llm_score
10650,Change email - Trial,Account,Low,0
2041,API Error 500 - Style,Technical,Low,2
8668,Login failed - Speak,Technical,Low,2
1114,Payment failed - Them,Billing,Low,0
13902,Login failed - Learn,Technical,Low,0
11963,Refund status - Strong,Billing,Low,1
11072,App crashing - Computer,Technical,Low,2
3002,Update credit card - Own,Billing,Low,0
19771,Data not syncing - Travel,Technical,Critical,2
8115,Screen freezes - South,Technical,Medium,2


In [ ]:
import os
from tqdm import tqdm

CHECKPOINT_FILE = "llm_scores_checkpoint.csv"
FINAL_FILE = "llm_scores.csv"
SAVE_EVERY = 500

In [ ]:
if os.path.exists(CHECKPOINT_FILE):

    checkpoint = pd.read_csv(
        CHECKPOINT_FILE
    )

    completed_ids = set(
        checkpoint["Ticket_ID"]
    )

    remaining_df = df[
        ~df["Ticket_ID"].isin(
            completed_ids
        )
    ].copy()

    results = checkpoint.to_dict(
        "records"
    )

    print(
        f"Completed: {len(checkpoint)}"
    )

else:

    remaining_df = df.copy()

    results = []

In [ ]:
counter = 0

for _, row in tqdm(
    remaining_df.iterrows(),
    total=len(remaining_df)
):

    try:

        score = predict_severity(
            row["Ticket_Subject"],
            row["Ticket_Description"],
            row["Issue_Category"],
            row["Ticket_Channel"]
        )

    except Exception:

        score = None

    results.append({

        "Ticket_ID":
        row["Ticket_ID"],

        "llm_score":
        score
    })

    counter += 1

    if counter % SAVE_EVERY == 0:

        pd.DataFrame(
            results
        ).to_csv(
            CHECKPOINT_FILE,
            index=False
        )

100%|██████████| 20000/20000 [1:40:04<00:00,  3.33it/s]


In [ ]:
final_df = pd.DataFrame(
    results
)

final_df.to_csv(
    "llm_scores.csv",
    index=False
)

print(
    final_df["llm_score"]
    .value_counts()
)

llm_score
0    12969
2     5087
1     1623
3     1153
Name: count, dtype: int64


In [ ]:
final_df.shape

(20832, 2)